# Alpha Benchmark Study (S&P 100 vs `^OEX`)

This notebook:

1. Loads daily data (2020-01-01 to 2024-12-31) for 100 S&P large-cap names using yfinance-backed `finTs`.
2. Plots universe return behavior.
3. Runs all alphas from `examples/alphas` through `FinStrat` + `FinBT`.
4. Compares each alpha against `^OEX` (S&P 100 benchmark).
5. Plots cross-correlation (`alphas <-> benchmark` and `alphas <-> alphas`) and performance metrics.


In [1]:
import re
import warnings
from dataclasses import dataclass
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from curl_cffi import requests as curl_requests

from shunya import FinBT, FinStrat, finTs
from examples.alphas import ALL_ALPHAS

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)

# Reuse one curl_cffi session for all network calls (wiki + yfinance).
session = curl_requests.Session(impersonate="chrome")


In [2]:
START_DATE = "2020-01-01"
END_DATE = "2024-12-31"
BENCHMARK = "^OEX"


def _fallback_symbols() -> list[str]:
    # Large-cap fallback list if online S&P 100 fetch fails.
    return [
        "AAPL", "MSFT", "AMZN", "NVDA", "GOOGL", "GOOG", "META", "BRK-B", "LLY", "AVGO",
        "JPM", "XOM", "V", "UNH", "COST", "PG", "MA", "HD", "MRK", "CVX",
        "ABBV", "PEP", "ADBE", "KO", "BAC", "WMT", "MCD", "CSCO", "TMO", "PFE",
        "CRM", "ACN", "ORCL", "ABT", "NFLX", "DHR", "WFC", "DIS", "LIN", "TXN",
        "AMGN", "VZ", "INTC", "PM", "UNP", "COP", "CAT", "NKE", "QCOM", "IBM",
        "GE", "NOW", "RTX", "HON", "SPGI", "AMD", "LOW", "AMAT", "INTU", "SBUX",
        "GS", "PLD", "BLK", "ELV", "T", "BKNG", "MDT", "GILD", "ADP", "LRCX",
        "C", "SYK", "TJX", "DE", "VRTX", "CI", "PANW", "MMC", "MU", "ADI",
        "SCHW", "ETN", "CB", "TMUS", "ISRG", "SLB", "LMT", "SO", "PGR", "REGN",
        "DUK", "MO", "ZTS", "USB", "CME", "EOG", "AON", "APD", "MCO", "CSX",
    ]


def _parse_sp100_from_wiki_html(html: str) -> list[str]:
    # Prefer table parsing when available.
    try:
        frames = pd.read_html(StringIO(html))
        for frame in frames:
            cols = [str(c) for c in frame.columns]
            if "Symbol" in cols:
                raw = frame["Symbol"].astype(str)
                return [x.strip().replace(".", "-") for x in raw if x and x != "nan"]
            if "Ticker symbol" in cols:
                raw = frame["Ticker symbol"].astype(str)
                return [x.strip().replace(".", "-") for x in raw if x and x != "nan"]
    except Exception:
        pass

    # Fallback: regex extraction from wiki HTML links.
    # Captures entries like /wiki/NYSE:IBM, /wiki/NASDAQ:AAPL, /wiki/BRK-B.
    pats = [
        r"/wiki/(?:NYSE|NASDAQ|NYSEARCA):([A-Z][A-Z0-9.\-]+)",
        r"/wiki/([A-Z]{1,5}(?:\.[A-Z])?)\"",
    ]
    found = []
    for p in pats:
        found.extend(re.findall(p, html))
    cleaned = []
    seen = set()
    for sym in found:
        s = sym.strip().replace(".", "-")
        if not s or len(s) > 6:
            continue
        if s in {"NYSE", "NASDAQ", "ETF", "SPX", "OEX"}:
            continue
        if s not in seen:
            seen.add(s)
            cleaned.append(s)
    return cleaned


def get_sp100_symbols(n: int = 100) -> list[str]:
    url = "https://en.wikipedia.org/wiki/S%26P_100"
    try:
        resp = session.get(url, timeout=30)
        resp.raise_for_status()
        syms = _parse_sp100_from_wiki_html(resp.text)
        if len(syms) < n:
            raise ValueError(f"Parsed only {len(syms)} symbols from wiki page")
    except Exception as exc:
        print(f"Falling back to static large-cap list due to: {exc}")
        syms = _fallback_symbols()

    unique = []
    seen = set()
    for s in syms:
        if s not in seen:
            unique.append(s)
            seen.add(s)

    # Preserve requested size even if parser returns extras.
    if len(unique) < n:
        extra = [x for x in _fallback_symbols() if x not in seen]
        unique.extend(extra)
    return unique[:n]


symbols = get_sp100_symbols(100)
print(f"Loaded {len(symbols)} symbols")
print(symbols[:20])


Falling back to static large-cap list due to: Failed to perform, curl: (60) SSL certificate problem: self signed certificate in certificate chain. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
Loaded 100 symbols
['AAPL', 'MSFT', 'AMZN', 'NVDA', 'GOOGL', 'GOOG', 'META', 'BRK-B', 'LLY', 'AVGO', 'JPM', 'XOM', 'V', 'UNH', 'COST', 'PG', 'MA', 'HD', 'MRK', 'CVX']


In [3]:
# Build panel with yfinance-backed provider (default in finTs).
# Pass curl_cffi session to yfinance path for better network robustness.
fts = finTs(
    START_DATE,
    END_DATE,
    symbols,
    session=session,
    feature_mode="ohlcv_only",
    strict_provider_universe=False,
    strict_ohlcv=True,
    strict_empty=False,
    attach_yfinance_classifications=False,
)

if fts.df.empty:
    raise ValueError(
        "Downloaded panel is empty. Check network/yfinance availability and retry."
    )

align_report = fts.align_universe(
    ("Open", "High", "Low", "Close", "Volume"),
    on_bad_ticker="drop",
)

print("Alignment summary:")
print(align_report.as_dict())

universe = list(fts.ticker_list)
print(f"Usable universe size after alignment: {len(universe)}")


Failed to get ticker 'MCO' reason: Failed to perform, curl: (60) SSL certificate problem: self signed certificate in certificate chain. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
Failed to get ticker 'AON' reason: Failed to perform, curl: (60) SSL certificate problem: self signed certificate in certificate chain. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
Failed to get ticker 'APD' reason: Failed to perform, curl: (60) SSL certificate problem: self signed certificate in certificate chain. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$APD: possibly delisted; no timezone found
Failed to get ticker 'BRK-B' reason: Failed to perform, curl: (60) SSL certificate problem: self signed certificate in certificate chain. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$BRK-B: possibly delisted; no timezone found
Failed to get ticker 'TJX' reason: Failed to perform, curl: (60) SSL c

KeyboardInterrupt: 

Failed to get ticker 'PGR' reason: Failed to perform, curl: (60) SSL certificate problem: self signed certificate in certificate chain. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$PGR: possibly delisted; no timezone found
Failed to get ticker 'MU' reason: Failed to perform, curl: (60) SSL certificate problem: self signed certificate in certificate chain. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$MU: possibly delisted; no timezone found
Failed to get ticker 'SPGI' reason: Failed to perform, curl: (60) SSL certificate problem: self signed certificate in certificate chain. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$SPGI: possibly delisted; no timezone found
Failed to get ticker 'SLB' reason: Failed to perform, curl: (60) SSL certificate problem: self signed certificate in certificate chain. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
$SLB: possibly delisted; no tim

In [ ]:
# Universe proxy return: equal-weight daily return of aligned Close series.
close_panel = (
    fts.df["Close"]
    .unstack("Ticker")
    .sort_index()
)

universe_rets = close_panel.pct_change().replace([np.inf, -np.inf], np.nan)
universe_eq_ret = universe_rets.mean(axis=1, skipna=True).dropna().rename("Universe_EQ")
universe_eq_cum = (1.0 + universe_eq_ret).cumprod().rename("Universe_EQ")

fig, ax = plt.subplots(figsize=(12, 4))
(universe_eq_cum - 1.0).plot(ax=ax, color="C0", linewidth=2)
ax.set_title("Equal-Weight Universe Cumulative Return (Aligned 100-name panel)")
ax.set_ylabel("Cumulative return")
ax.set_xlabel("Date")
plt.show()


In [ ]:
# Benchmark (^OEX) returns from yfinance using same curl_cffi session.
bench_px = yf.download(
    BENCHMARK,
    start=START_DATE,
    end=pd.Timestamp(END_DATE) + pd.Timedelta(days=1),
    auto_adjust=True,
    progress=False,
    session=session,
)

if bench_px.empty:
    raise ValueError("Benchmark download returned empty frame for ^OEX")

bench_close = bench_px["Close"].rename("OEX_Close")
bench_ret = bench_close.pct_change().dropna().rename("OEX")
bench_cum = (1.0 + bench_ret).cumprod().rename("OEX")

fig, ax = plt.subplots(figsize=(12, 4))
(bench_cum - 1.0).plot(ax=ax, color="black", linewidth=2)
ax.set_title("^OEX Cumulative Return")
ax.set_ylabel("Cumulative return")
ax.set_xlabel("Date")
plt.show()


In [ ]:
@dataclass
class AlphaRun:
    name: str
    equity: pd.Series
    returns: pd.Series
    metrics: dict


runs: dict[str, AlphaRun] = {}

for alpha_name, alpha_fn in ALL_ALPHAS.items():
    fs = FinStrat(
        fts,
        alpha_fn,
        neutralization="market",
        truncation=0.01,
    )
    bt = FinBT(
        fs,
        fts,
        cash=1_000_000.0,
        commission=0.0005,
        slippage_pct=0.0005,
    ).run()
    out = bt.results(show=False)

    eq = out["equity_curve"]["Equity"].astype(float).rename(alpha_name)
    ret = eq.pct_change().dropna().replace([np.inf, -np.inf], np.nan).dropna().rename(alpha_name)

    runs[alpha_name] = AlphaRun(
        name=alpha_name,
        equity=eq,
        returns=ret,
        metrics=out["metrics"],
    )

print(f"Completed backtests for {len(runs)} alphas: {list(runs.keys())}")


In [ ]:
# Align alpha returns with benchmark and build cumulative returns table.
alpha_returns = pd.concat([r.returns for r in runs.values()], axis=1, join="inner")
all_returns = alpha_returns.join(bench_ret, how="inner").dropna(how="any")

all_cum = (1.0 + all_returns).cumprod()

fig, ax = plt.subplots(figsize=(14, 6))
for col in all_cum.columns:
    lw = 2.5 if col == "OEX" else 1.6
    alpha = 1.0 if col == "OEX" else 0.9
    all_cum[col].sub(1.0).plot(ax=ax, linewidth=lw, alpha=alpha, label=col)
ax.set_title("Cumulative Return: Example Alphas vs ^OEX")
ax.set_ylabel("Cumulative return")
ax.set_xlabel("Date")
ax.legend(loc="best", ncol=2)
plt.show()

print(all_returns.shape)
all_returns.tail()


In [ ]:
# Correlation matrix: all alphas + benchmark.
corr = all_returns.corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Return Correlation: Alphas and ^OEX")
plt.show()

corr


In [ ]:
def perf_stats(ret: pd.Series) -> dict:
    ret = ret.dropna()
    if ret.empty:
        return {
            "total_return_pct": np.nan,
            "cagr_pct": np.nan,
            "ann_vol_pct": np.nan,
            "sharpe": np.nan,
            "max_drawdown_pct": np.nan,
        }

    cum = (1.0 + ret).cumprod()
    total_return = cum.iloc[-1] - 1.0
    n_years = max((cum.index[-1] - cum.index[0]).days / 365.25, 1e-9)
    cagr = cum.iloc[-1] ** (1.0 / n_years) - 1.0

    ann_vol = ret.std(ddof=1) * np.sqrt(252)
    sharpe = (ret.mean() / ret.std(ddof=1) * np.sqrt(252)) if ret.std(ddof=1) > 0 else np.nan

    drawdown = cum / cum.cummax() - 1.0
    max_dd = drawdown.min()

    return {
        "total_return_pct": total_return * 100.0,
        "cagr_pct": cagr * 100.0,
        "ann_vol_pct": ann_vol * 100.0,
        "sharpe": sharpe,
        "max_drawdown_pct": max_dd * 100.0,
    }


metric_rows = []
for name in alpha_returns.columns:
    stats = perf_stats(all_returns[name])
    stats["corr_to_oex"] = all_returns[name].corr(all_returns["OEX"])
    stats["name"] = name
    metric_rows.append(stats)

bench_stats = perf_stats(all_returns["OEX"])
bench_stats["corr_to_oex"] = 1.0
bench_stats["name"] = "OEX"
metric_rows.append(bench_stats)

metrics_df = pd.DataFrame(metric_rows).set_index("name").sort_values("sharpe", ascending=False)
metrics_df


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

plot_specs = [
    ("total_return_pct", "Total Return (%)"),
    ("cagr_pct", "CAGR (%)"),
    ("sharpe", "Sharpe"),
    ("max_drawdown_pct", "Max Drawdown (%)"),
]

for ax, (col, title) in zip(axes.ravel(), plot_specs):
    view = metrics_df[[col]].sort_values(col, ascending=False)
    colors = ["black" if idx == "OEX" else "C0" for idx in view.index]
    ax.bar(view.index, view[col], color=colors, alpha=0.85)
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# Alpha-only correlation (without benchmark), useful for diversification analysis.
alpha_corr = alpha_returns.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    alpha_corr,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Alpha-to-Alpha Return Correlation")
plt.show()
